In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

In [1]:
# 감정분석을 위한 모델과 토크나이저를 가져온다. => pipeline 방식이 아니다.

In [6]:
model_name = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [9]:
text = 'I love this movie! It was fantastic!'

In [10]:
# 1. 토크나이저를 이용해서, 문장을 토큰으로 만든다.

In [11]:
inputs = tokenizer(text, return_tensors='pt')

In [12]:
inputs

{'input_ids': tensor([[  101,  1045,  2293,  2023,  3185,   999,  2009,  2001, 10392,   999,
           102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [18]:
# 모델 실행
result = model(**inputs)

In [19]:
logits = result.logits

In [20]:
logits

tensor([[-4.3242,  4.6727]], grad_fn=<AddmmBackward0>)

In [ ]:
# 로짓을 클래스로 매핑한다.
# 소프트맥스를 적용해서 확률로 계산한다.

In [21]:
import torch

In [23]:
probs = torch.nn.functional.softmax(logits, dim=-1)

In [24]:
probs

tensor([[1.2378e-04, 9.9988e-01]], grad_fn=<SoftmaxBackward0>)

In [28]:
predicted_classes = torch.argmax(probs, dim=-1).tolist()

In [29]:
predicted_classes

[1]

In [30]:
# Config 도 가져온다.
from transformers import AutoConfig

In [31]:
config = AutoConfig.from_pretrained(model_name)

In [32]:
config

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.56.1",
  "vocab_size": 30522
}

In [45]:
config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

In [44]:
int(predicted_classes[0])

1

In [34]:
config.id2label[predicted_classes[0]]

'POSITIVE'

In [35]:
# 두개 이상의 데이터를 처리할때.

In [36]:
texts = ["I love this movie! It was fantastic!",
         "This is the worst experience I have ever had.",
         '''In today's fast-paced and highly interconnected world, technology has revolutionized nearly every aspect of human life, from the way we communicate and work to how we access information, entertain ourselves, and even manage our daily routines, with smartphones, artificial intelligence, cloud computing, and the internet playing crucial roles in shaping modern society by enabling instant global communication, facilitating remote work, streamlining complex processes, and providing an unprecedented level of convenience that previous generations could hardly have imagined; however, while these advancements have brought countless benefits, such as increased efficiency, improved accessibility to education and healthcare, and the ability to stay connected with friends and family regardless of physical distance, they have also introduced new challenges and concerns, including privacy risks, cybersecurity threats, social isolation, and the potential for misinformation to spread rapidly across digital platforms, making it essential for individuals, businesses, and governments to strike a delicate balance between embracing innovation and implementing safeguards to protect personal data, ensure online security, and promote ethical technology use, all while fostering a digital landscape that encourages creativity, inclusivity, and meaningful human interactions rather than dependency on screens and algorithms to dictate every aspect of daily life.''']


In [37]:
# 문장이 여러개일때는 문장의 길이가 다 다르므로, 파라미터 설정을 해줘야 한다.

In [38]:
inputs = tokenizer(texts , return_tensors='pt', padding=True, max_length=128, truncation=True)

In [39]:
inputs

{'input_ids': tensor([[  101,  1045,  2293,  2023,  3185,   999,  2009,  2001, 10392,   999,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [48]:
logits = model(**inputs).logits

In [49]:
probs = torch.nn.functional.softmax(logits, dim=-1)

In [50]:
probs

tensor([[1.2378e-04, 9.9988e-01],
        [9.9976e-01, 2.3730e-04],
        [3.2476e-04, 9.9968e-01]], grad_fn=<SoftmaxBackward0>)

In [53]:
predicted_classes = torch.argmax(probs, dim=-1).tolist()

In [54]:
predicted_classes

[1, 0, 1]

In [55]:
for  id  in predicted_classes :
  print( config.id2label[id] )

POSITIVE
NEGATIVE
POSITIVE


In [57]:
texts = [ "디자인이 세련되고 사용감도 좋아서 매일 쓰고 있어요." ,
         "배송이 너무 늦게 와서 기다리느라 지쳤습니다. 제품 포장도 허술해서 이미 모서리가 살짝 손상된 상태로 도착했는데, 교환 요청을 하려고 고객센터에 전화했더니 연결이 잘 되지 않아 불편했습니다. 가격에 비해 만족도가 낮아 다시 구매할 생각은 없습니다.",
         "제품 설명과 달라서 실망했고, 품질도 별로였어요.",
         "상품 상태가 아주 깔끔하게 도착했어요. 실제로 사용해보니 설명대로 기능이 잘 작동하고 편리했습니다. 특히 배터리 지속 시간이 길어서 하루 종일 사용해도 문제없었어요."
]

In [56]:
model_name = 'WhitePeak/bert-base-cased-Korean-sentiment'

In [59]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

In [60]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [61]:
config = AutoConfig.from_pretrained(model_name)

In [62]:
texts

['디자인이 세련되고 사용감도 좋아서 매일 쓰고 있어요.',
 '배송이 너무 늦게 와서 기다리느라 지쳤습니다. 제품 포장도 허술해서 이미 모서리가 살짝 손상된 상태로 도착했는데, 교환 요청을 하려고 고객센터에 전화했더니 연결이 잘 되지 않아 불편했습니다. 가격에 비해 만족도가 낮아 다시 구매할 생각은 없습니다.',
 '제품 설명과 달라서 실망했고, 품질도 별로였어요.',
 '상품 상태가 아주 깔끔하게 도착했어요. 실제로 사용해보니 설명대로 기능이 잘 작동하고 편리했습니다. 특히 배터리 지속 시간이 길어서 하루 종일 사용해도 문제없었어요.']

In [63]:
inputs = tokenizer(texts, return_tensors='pt', padding=True, max_length=140, truncation=True)

In [64]:
inputs

{'input_ids': tensor([[   101,   9122,  86150,  10739,   9435, 101440,  29208,   9405,  24974,
         105197,  12092,   9685,  16985,  12424,   9258,  18392,   9511,  11664,
          45893,  48549,    119,    102,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0],
        [   101,   9330, 119057,  10739,   9004,  32537,   9047,  14153,   9590,
          12424,   8932,  119

In [67]:
logits = model(**inputs).logits

In [68]:
logits

tensor([[-2.6011,  2.2324],
        [ 3.0323, -2.8744],
        [ 2.9684, -2.8548],
        [-2.5900,  2.2183]], grad_fn=<AddmmBackward0>)

In [69]:
probs = torch.nn.functional.softmax(logits, dim=-1)

In [70]:
probs

tensor([[0.0079, 0.9921],
        [0.9973, 0.0027],
        [0.9971, 0.0029],
        [0.0081, 0.9919]], grad_fn=<SoftmaxBackward0>)

In [71]:
config

BertConfig {
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "dtype": "float32",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "transformers_version": "4.56.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 119547
}

In [72]:
labels = {0 : '부정', 1 : '긍정'}

In [75]:
torch.argmax(probs, dim=-1)

tensor([1, 0, 0, 1])

In [76]:
probs.argmax(axis = 1)

tensor([1, 0, 0, 1])

In [77]:
predicted_classes = probs.argmax(axis = 1)

In [78]:
predicted_classes

tensor([1, 0, 0, 1])

In [83]:
for id in predicted_classes.tolist() :
  print( labels[id] )

긍정
부정
부정
긍정


In [84]:
texts

['디자인이 세련되고 사용감도 좋아서 매일 쓰고 있어요.',
 '배송이 너무 늦게 와서 기다리느라 지쳤습니다. 제품 포장도 허술해서 이미 모서리가 살짝 손상된 상태로 도착했는데, 교환 요청을 하려고 고객센터에 전화했더니 연결이 잘 되지 않아 불편했습니다. 가격에 비해 만족도가 낮아 다시 구매할 생각은 없습니다.',
 '제품 설명과 달라서 실망했고, 품질도 별로였어요.',
 '상품 상태가 아주 깔끔하게 도착했어요. 실제로 사용해보니 설명대로 기능이 잘 작동하고 편리했습니다. 특히 배터리 지속 시간이 길어서 하루 종일 사용해도 문제없었어요.']

In [85]:
! pip install datasets

In [86]:
from datasets import load_dataset

In [87]:
dataset = load_dataset('imdb')

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [88]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [89]:
model_name = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'

In [90]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [91]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [92]:
config = AutoConfig.from_pretrained(model_name)

In [95]:
type( dataset.data )

dict

In [96]:
dataset.data.keys()

dict_keys(['train', 'test', 'unsupervised'])

In [104]:
dataset.data['train']['label'][2]

<pyarrow.Int64Scalar: 0>

In [ ]:
def tokenize_function(data) :
  return tokenizer(data['text'], padding=True, max_length=128, truncation=True)